In [ ]:
# ==============================================================================
# Counterfactual Explanations Pipeline (Paper 2) - Complete & Stable Version
# ==============================================================================
import os
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import dice_ml

import os
from pyprojroot import here

warnings.filterwarnings('ignore')

# ตั้งค่าโฟลเดอร์สำหรับเซฟภาพ
OUT_DIR = 'figures_paper2'
os.makedirs(OUT_DIR, exist_ok=True)

base_path = here()
file_path = os.path.join(base_path, "datas", "SFProgramDataPanal_with_clusters.csv")

df = pd.read_csv(file_path)

In [ ]:
# ------------------------------------------------------------------------------
# STEP 1: Data Preparation (อิง Cluster เดิมจาก Paper 1 อย่างสมบูรณ์)
# ------------------------------------------------------------------------------
print("--- STEP 1: Loading Data & Stable Clusters ---")

df = df.replace('.', np.nan)
df_y0 = df[df['Year'] == 0].copy()

CLUSTER_FEATS = [
    'age', 'edu', 'agri_long', 'irriga', 'loan',
    'Avg_ProdManage', 'Avg_InputManage', 'Avg_Tech',
    'Avg_Ana&Plan', 'Avg_Mkting', 'Avg_Network',
    'Ave_ProdRisk', 'Ave_InputRisk', 'Ave_MktRisk', 'Ave_FinRisk'
]

# ดึงเฉพาะคอลัมน์ที่ใช้งานและลบค่าสูญหาย
df_cf = df_y0[['id', 'Cluster'] + CLUSTER_FEATS].apply(pd.to_numeric, errors='coerce').dropna().copy()
X_df = df_cf[CLUSTER_FEATS].copy()
y_df = df_cf['Cluster'].copy()

print(f"Data Loaded: {len(X_df)} records. Clusters Distribution:\n{y_df.value_counts().to_string()}")



In [ ]:
# ------------------------------------------------------------------------------
# STEP 2: Train Black-Box Model (RF)
# ------------------------------------------------------------------------------
print("\n--- STEP 2: Training Black-Box Classifier ---")
RF_PARAMS = dict(n_estimators=500, min_samples_leaf=10, max_features='sqrt', random_state=42, n_jobs=-1)
rf_final = RandomForestClassifier(**RF_PARAMS)
rf_final.fit(X_df, y_df)

# ตรวจสอบความแม่นยำเบื้องต้น (Training Acc)
train_acc = accuracy_score(y_df, rf_final.predict(X_df))
print(f"✓ Base Classifier Trained (Accuracy on Full Data: {train_acc:.4f})")

In [ ]:
# ------------------------------------------------------------------------------
# STEP 3: Setup DiCE (Genetic Method & Monotonic Constraints)
# ------------------------------------------------------------------------------
print("\n--- STEP 3: Initializing DiCE (Genetic) ---")
immutable_feats = ['age', 'edu', 'agri_long']
features_to_vary = [f for f in CLUSTER_FEATS if f not in immutable_feats]

d_data = dice_ml.Data(dataframe=pd.concat([X_df, y_df], axis=1), continuous_features=CLUSTER_FEATS, outcome_name='Cluster')
d_model = dice_ml.Model(model=rf_final, backend="sklearn", model_type="classifier")
# ใช้ Genetic Algorithm ตามคำแนะนำของ Reviewer
exp_dice = dice_ml.Dice(d_data, d_model, method="genetic")

def get_permitted_range(farmer_profile):
    """ สร้างเงื่อนไขห้ามสกิลลดลง และป้องกัน Error ขอบเขตแคบเกินไป """
    ranges = {}
    for feat in features_to_vary:
        val = float(farmer_profile[feat].values[0])
        if 'Avg_' in feat:
            ranges[feat] = [val, val + 0.01] if val >= 5.0 else [val, 5.0]
        else:
            ranges[feat] = [X_df[feat].min(), X_df[feat].max()]
    return ranges



In [ ]:
# ------------------------------------------------------------------------------
# STEP 4: Batch Evaluation (0->1 และ 1->2) & Metrics Calculation
# ------------------------------------------------------------------------------
print("\n--- STEP 4: Batch Processing & Metrics ---")
COURSE_MAPPING = {
    'Avg_ProdManage': 'Production Planning & Management',
    'Avg_InputManage': 'Input & Resource Optimization',
    'Avg_Tech': 'Technology Adoption',
    'Avg_Ana&Plan': 'Agri-Business Analysis',
    'Avg_Mkting': 'Marketing & Value Addition',
    'Avg_Network': 'Network Building'
}

def run_batch_evaluation(source_cluster, target_cluster, limit=10):
    """ 
    หมายเหตุ: ตั้ง limit=10 ไว้ให้เทสต์โค้ดผ่านก่อน 
    เวลาทำ Paper จริง ให้ลบพารามิเตอร์ , limit=10 ออกตอนเรียกฟังก์ชันครับ 
    """
    samples = X_df[y_df == source_cluster]
    if limit: samples = samples.head(limit) 
    
    print(f"Processing Cluster {source_cluster} -> {target_cluster} (Testing {len(samples)} farmers)...")
    results, action_plans = [], []
    
    start_time = time.time()
    for idx, (index, row) in enumerate(samples.iterrows()):
        farmer_profile = pd.DataFrame([row])
        ranges = get_permitted_range(farmer_profile)
        
        try:
            # Generate options to pick the best one
            cf_res = exp_dice.generate_counterfactuals(
                farmer_profile, total_CFs=5, desired_class=target_cluster,
                features_to_vary=features_to_vary, permitted_range=ranges,
                posthoc_sparsity_param=0.1
            )
            
            if cf_res and len(cf_res.cf_examples_list) > 0 and cf_res.cf_examples_list[0].final_cfs_df is not None:
                orig_df = cf_res.cf_examples_list[0].test_instance_df
                cfs_df = cf_res.cf_examples_list[0].final_cfs_df
                
                best_sparsity, best_l1, best_cf_row = 999, 999.0, None
                
                # กรองคำตอบที่ดีที่สุด (Sparsity ต่ำสุด, L1 ต่ำสุด และ Validity = True)
                for _, cf_row in cfs_df.iterrows():
                    # เช็ก Validity ยืนยันกับโมเดลอีกรอบ (ตามคอมเมนต์ Reviewer)
                    is_valid = rf_final.predict(pd.DataFrame([cf_row])[CLUSTER_FEATS])[0] == target_cluster
                    if not is_valid: continue
                        
                    changed_feats, l1_dist = 0, 0.0
                    for feat in features_to_vary:
                        diff = float(cf_row[feat]) - float(orig_df[feat].values[0])
                        if diff > 0.01: # ตัด Noise ของ Float
                            changed_feats += 1
                            l1_dist += diff
                            
                    if changed_feats < best_sparsity or (changed_feats == best_sparsity and l1_dist < best_l1):
                        best_sparsity = changed_feats
                        best_l1 = l1_dist
                        best_cf_row = cf_row
                
                if best_cf_row is not None:
                    results.append({'valid': 1, 'sparsity': best_sparsity, 'l1': best_l1})
                    # สกัดหลักสูตร
                    for feat, course in COURSE_MAPPING.items():
                        diff = float(best_cf_row[feat]) - float(orig_df[feat].values[0])
                        if diff >= 0.1:
                            action_plans.append({
                                'Farmer_ID': index, 
                                'Path': f'C{source_cluster} -> C{target_cluster}',
                                'Course': course, 'Increment': diff
                            })
                    continue
        except Exception:
            pass # Fallback ไปจดเป็น Failed ด้านล่าง
        
        results.append({'valid': 0, 'sparsity': np.nan, 'l1': np.nan})
        
    res_df = pd.DataFrame(results)
    suc_rate = res_df['valid'].mean() * 100
    avg_sp = res_df[res_df['valid'] == 1]['sparsity'].mean()
    avg_l1 = res_df[res_df['valid'] == 1]['l1'].mean()
    exec_t = time.time() - start_time
    
    print(f" ✓ Result: Success {suc_rate:.1f}% | Avg Sparsity: {avg_sp:.2f} | Avg L1: {avg_l1:.2f} | Time: {exec_t:.1f}s")
    return action_plans

# รันลูปสำหรับทั้ง 2 ทิศทาง (แก้ไข limit=None เพื่อรันข้อมูลทั้งหมดตอนทำ Paper) สำหรับทดสอบโค้ดผ่านก่อน ให้ตั้ง limit=10 ไว้ครับ แต่ถ้าจะรันจริงให้ใส่ limit=None เพื่อรันข้อมูลทั้งหมด
plans_0_to_1 = run_batch_evaluation(source_cluster=0, target_cluster=1, limit=10)
plans_1_to_2 = run_batch_evaluation(source_cluster=1, target_cluster=2, limit=10)




In [ ]:
# ------------------------------------------------------------------------------
# STEP 5: Visualizing Global Explanations (Actionability)
# ------------------------------------------------------------------------------
print("\n--- STEP 5: Visualizing Results ---")
all_plans = pd.DataFrame(plans_0_to_1 + plans_1_to_2)

if not all_plans.empty:
    plt.figure(figsize=(11, 6))
    ax = sns.countplot(data=all_plans, y='Course', hue='Path', palette='Set2')
    
    plt.title('Recommended Training Programs by Cluster Transition Path', fontsize=14, pad=15)
    plt.xlabel('Frequency (Number of Interventions)', fontsize=12)
    plt.ylabel('Skill Domain / Training Course', fontsize=12)
    plt.legend(title='Transition Path')
    plt.tight_layout()
    
    save_path = f"{OUT_DIR}/fig_training_recommendations.png"
    plt.savefig(save_path, dpi=300)
    print(f"✓ Visualization saved to: {save_path}")
    plt.show()
else:
    print("✕ No action plans were generated to visualize.")

In [ ]:
all_plans.head()